# 02 — Nấc 2: Prompt engineering

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/vinumqa-numerical-reasoning/blob/main/notebooks/02_prompt_engineering.ipynb)

**Cần GPU.** Khoảng 20–30 phút.

## Mục đích

Đo xem **prompt có cấu trúc** mang lại bao nhiêu, khi mọi thứ khác giữ nguyên: cùng model
Qwen3-8B chưa fine-tune, cùng `temperature`, cùng tập test, chỉ đổi system prompt.

## Prompt ở nấc này

Bản đầy đủ của dự án (`vinumqa/_prompt_text.py`, import nguyên văn nên không bao
giờ lệch bản). So với nấc 1 nó thêm bốn nhóm nội dung:

| Thêm gì | Nhắm vào lỗi nào |
|---|---|
| Giải thích ý nghĩa và cách dùng từng phép toán | chọn sai phép |
| **Ánh xạ từ khoá tiếng Việt → phép toán** ("gấp bao nhiêu lần" → `divide`, "cao nhất" → `table_max`, …) | không biết dịch câu hỏi thành phép tính |
| Quy tắc bắt buộc: không lồng phép, không `multiply(#n,100)`, phải quy đổi đơn vị *trong* program | program không chạy được, sai bậc độ lớn |
| 2 ví dụ mẫu đầy đủ | sai định dạng đầu ra |

## Đây là nấc quan trọng về mặt phương pháp

Mốc tham chiếu cho thấy prompt engineering một mình đã đưa Qwen3-8B vượt 50 % PA. Nếu nấc này
không tái lập được mức đó thì phải dừng lại kiểm tra môi trường trước khi đi tiếp.

## §1. Môi trường

In [ ]:
%%capture
# Khối cài đặt giữ NGUYÊN của reference/original_notebooks/inference_with_difference_models.ipynb
# để môi trường khớp với lần chạy tham chiếu.
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install vllm

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
ON_KAGGLE = os.path.isdir("/kaggle/input")
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
elif ON_KAGGLE:
    for _r, _d, _f in os.walk("/kaggle/input"):
        if "vinumqa" in _d and "data" in _d:
            REPO_DIR = _r; _pinned = True; break
    OUTPUT_DIR = "/kaggle/working/vinumqa_runs"
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

LADDER = [
    ("01_plain",           "Nấc 1 — inference thông thường"),
    ("02_prompt_eng",      "Nấc 2 — + prompt engineering"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
]
LADDER_LABEL = dict(LADDER)


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, csv, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "csv":   f"{stage}_program.csv",
        "meta":  f"{stage}_meta.json"}[kind])


def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: 3 file chuẩn + 1 file output thô."""
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_details_csv(rows, stage_path(stage, "csv"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'csv')}   ← 6 cột, mở bằng Excel được")
        print(f"      {stage_path(stage, 'meta')}")


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else ("Kaggle" if ON_KAGGLE else "local")
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + csv + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# ═══ Self-test: chạy TRƯỚC khi tốn GPU ═══
# Cell này đỏ thì dừng lại — mọi con số PA/EA sau đó sẽ vô nghĩa.
_ok = sum(dsl.check_ea(dsl.execute_program(s["qa"]["program"], s.get("table") or []),
                       s["qa"].get("exe_ans")) for s in test_all)
print(f"[SELF-TEST] executor tái tạo exe_ans trên test: {_ok}/{len(test_all)}")
assert _ok / len(test_all) > 0.99, "Executor không tái tạo được nhãn vàng — DỪNG."
assert dsl.execute_program("divide(5310, add(1, 0.15))", []) is None   # lồng nhau
assert dsl.check_pa("add(1, 2)", "add(2, 1)")[0]                       # giao hoán
assert dsl.check_ea(0.6066481994, "0.60665")                           # làm tròn 5 chữ số
print("[SELF-TEST] ✅ executor / PA / EA đạt")

## §2. Model

In [ ]:
# ═══════════════ MODEL — Qwen3-8B 4-bit ═══════════════
# Tham số lấy từ reference/original_notebooks/inference_with_difference_models.ipynb:
#   load_in_4bit=True, fast_inference=True, temperature=0.1, max_tokens=3000
MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU. Runtime → Change runtime type → L4 GPU.")
_GPU, _VRAM = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3
_CC = torch.cuda.get_device_capability(0)
if _CC[0] < 7:
    raise RuntimeError(f"{_GPU} (CC {_CC[0]}.{_CC[1]}) không chạy được vLLM. "
                       f"Trên Kaggle chọn 'GPU T4 x2', đừng chọn P100.")

TEMPERATURE, MAX_TOKENS = 0.1, 3000        # giữ đúng mốc tham chiếu
REPETITION_PENALTY = 1.0
BATCH_SIZE = 500                           # như notebook cũ; giảm nếu OOM

if _VRAM < 18:                             # T4 15GB
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 8192, 0.90, 16
    MAX_TOKENS, BATCH_SIZE = 1536, 32
elif _VRAM < 30:                           # L4 24GB
    # 13500 chứ không phải 12000: để ngân sách prompt (13500-3000=10500) vượt xa
    # prompt bước 2 dài nhất (~9k token) → L4 KHÔNG phải cắt ngữ cảnh, nhờ vậy
    # kết quả trên L4 và A100 so sánh trực tiếp được với nhau.
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 13500, 0.88, 24
    BATCH_SIZE = 128
elif _VRAM < 60:                           # A100 40GB — như notebook gốc
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 15000, 0.85, 64
else:                                      # A100 80GB — còn dư, tăng song song
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 15000, 0.85, 128
DTYPE = torch.float16 if _CC[0] < 8 else None

print(f"[GPU] {_GPU} | {_VRAM:.1f} GB | CC {_CC[0]}.{_CC[1]}")
print(f"[CFG] max_seq={MAX_SEQ_LENGTH} max_tokens={MAX_TOKENS} temp={TEMPERATURE} "
      f"batch={BATCH_SIZE}")
if _VRAM < 18:
    print("[CFG] ⚠ T4: đã hạ max_tokens xuống 1536 → kết quả KHÔNG so trực tiếp "
          "được với máy dùng 3000. Nên chạy trên L4.")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import unsloth
from unsloth import FastLanguageModel
from vllm import SamplingParams

torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"[MODEL] Đang tải {MODEL_NAME} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = MODEL_NAME,
    dtype                  = DTYPE,
    max_seq_length         = MAX_SEQ_LENGTH,
    load_in_4bit           = True,
    fast_inference         = True,
    gpu_memory_utilization = GPU_MEM_UTIL,
    max_num_seqs           = MAX_NUM_SEQS,
)

SAMPLING = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                          repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
LORA_REQUEST = None          # nấc 3 trở đi có thể gán adapter đã SFT vào đây

def generate(prompts, sampling_params=None, desc=None, batch_size=None):
    """Sinh theo lô qua vLLM — như vòng lặp trong notebook cũ."""
    if not prompts:
        return []
    sp = sampling_params or SAMPLING
    bs = batch_size or BATCH_SIZE
    outs, t0 = [], time.time()
    nb = (len(prompts) + bs - 1) // bs
    for i in range(nb):
        chunk = prompts[i*bs:(i+1)*bs]
        kw = {"sampling_params": sp}
        if LORA_REQUEST is not None:
            kw["lora_request"] = LORA_REQUEST
        outs.extend(o.outputs[0].text for o in model.fast_generate(chunk, **kw))
        if desc:
            el = time.time() - t0
            print(f"    {desc}: lô {i+1}/{nb} | {el:.0f}s | "
                  f"ETA {el/(i+1)*(nb-i-1):.0f}s", end="\r")
    gc.collect(); torch.cuda.empty_cache()
    if desc:
        print(f"    {desc}: xong {len(prompts)} prompt trong {time.time()-t0:.0f}s" + " "*16)
    return outs

_t = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"[MODEL] ✅ sẵn sàng | VRAM {_t - torch.cuda.mem_get_info()[0]/1024**3:.1f}/{_t:.1f} GB")
_ = generate(["xin chào"], SamplingParams(temperature=0, max_tokens=4))
print("[WARMUP] ✅")

## §3. Prompt có cấu trúc

In [ ]:
prompt_kit = PromptKit(REPO_DIR, tokenizer=tokenizer, model_name=MODEL_NAME)
PROMPT_LEVEL = "engineered"
USE_SELFEVAL = False
if PROMPT_LEVEL == "selfeval":
    PROMPT_LEVEL = "engineered"

print(f"[PROMPT] mức = {PROMPT_LEVEL} | self-eval = {USE_SELFEVAL}")
print(f"         plain={len(prompt_kit.PLAIN_SYSTEM_PROMPT)} ký tự | "
      f"engineered={len(prompt_kit.ENGINEERED_SYSTEM_PROMPT)} | "
      f"self-eval={len(prompt_kit.SELF_EVAL_SYSTEM_PROMPT)}")

# Đo bằng tokenizer THẬT trên 40 mẫu có ngữ cảnh DÀI NHẤT
BUDGET = MAX_SEQ_LENGTH - MAX_TOKENS
_clen = lambda s: (len(" ".join(s.get("pre_text") or [])) +
                   len(" ".join(s.get("post_text") or [])) + len(str(s.get("table") or "")))
_probe = sorted(test_all, key=_clen, reverse=True)[:40]
_bul = "\n".join(["- Khi hỏi tốc độ tăng trưởng, dùng subtract(gia_tri_moi, gia_tri_cu), "
                  "divide(#0, gia_tri_cu)."] * 7)
_prev = "Phân tích chi tiết. " * 150 + "\n```plaintext\nprogram: divide(1,2)\nanswer: 0.5\n```"

def _measure():
    a = [len(tokenizer(prompt_kit.step1(s, _bul, level=PROMPT_LEVEL)).input_ids)
         for s in _probe]
    b = ([len(tokenizer(prompt_kit.step2(s, _prev, _bul)).input_ids) for s in _probe]
         if USE_SELFEVAL else [0])
    return a, b

_a, _b = _measure()
print(f"[PROMPT] (40 mẫu dài nhất) step1 max={max(_a)} | step2 max={max(_b)} | "
      f"ngân sách={BUDGET}")

if max(max(_a), max(_b)) > BUDGET:
    prompt_kit.max_prev_chars = 3000
    _cap = _clen(_probe[0])
    for _ in range(6):
        _cap = int(_cap * 0.80)
        prompt_kit.max_ctx_chars = max(1200, _cap)
        _a, _b = _measure()
        if max(max(_a), max(_b)) <= BUDGET:
            break
    assert max(max(_a), max(_b)) <= BUDGET, "Không cắt đủ — giảm MAX_TOKENS hoặc dùng GPU lớn hơn."
    _hit = sum(1 for s in test_all if _clen(s) > prompt_kit.max_ctx_chars)
    print(f"[PROMPT] ⚠ đã bật cắt ngữ cảnh (max_ctx_chars={prompt_kit.max_ctx_chars}); "
          f"{_hit}/{len(test_all)} mẫu bị cắt ({_hit/len(test_all)*100:.1f}%)")
    print(f"[PROMPT]   GHI LẠI con số này khi báo cáo.")
else:
    print("[PROMPT] ✅ mọi prompt đều lọt ngân sách, không cần cắt")

In [ ]:
# So sánh trực tiếp hai system prompt
print(f"{'':<14}{'ký tự':>8}{'token':>8}")
for _name, _p in [("plain", prompt_kit.PLAIN_SYSTEM_PROMPT),
                  ("engineered", prompt_kit.ENGINEERED_SYSTEM_PROMPT)]:
    print(f"{_name:<14}{len(_p):>8}{len(tokenizer(_p).input_ids):>8}")

print(f"\n── Những phần CHỈ CÓ ở prompt engineered ──")
for _kw, _desc in [("HƯỚNG DẪN CHỌN PHÉP TOÁN THEO TỪ KHÓA", "ánh xạ từ khoá → phép toán"),
                   ("VÍ DỤ", "ví dụ mẫu"),
                   ("không được lồng", "cấm phép lồng nhau"),
                   ("multiply(#0, 100)", "cấm nhân 100 để ra phần trăm"),
                   ("quy đổi", "bắt buộc quy đổi đơn vị trong program")]:
    _in_plain = _kw.lower() in prompt_kit.PLAIN_SYSTEM_PROMPT.lower()
    _in_eng = _kw.lower() in prompt_kit.ENGINEERED_SYSTEM_PROMPT.lower()
    print(f"  {_desc:<42} plain={'có' if _in_plain else 'KHÔNG':<6} "
          f"engineered={'có' if _in_eng else 'KHÔNG'}")

## §4. Chạy trên tập test

In [ ]:
# (phần đọc/ghi đã nằm ở cell đầu)

In [ ]:
STAGE = "02_prompt_eng"
print(f"\n{'═'*74}\n  NẤC: {STAGE} | prompt={PROMPT_LEVEL} | self-eval={USE_SELFEVAL}"
      f" | {len(test_all)} mẫu\n{'═'*74}")

_t0 = time.time()
rows = pipeline.run_pipeline(
    test_all, prompt_kit, generate,
    prompt_level=PROMPT_LEVEL, use_selfeval=USE_SELFEVAL,
    sp_step1=SAMPLING, sp_step2=SAMPLING, desc=STAGE)
metrics = pipeline.summarize(rows, STAGE)
metrics["minutes"] = round((time.time() - _t0) / 60, 1)
pipeline.print_summary(metrics)
print(f"\n  Thời gian: {metrics['minutes']} phút")

## §5. Prompt engineering mang lại bao nhiêu

So sánh theo cặp với nấc 1 trên **cùng 497 mẫu**. Vì là dữ liệu cặp nên kiểm định đúng là
**McNemar**: chỉ nhìn các mẫu hai nấc bất đồng, đếm `b` (chỉ nấc 1 đúng) và `c` (chỉ nấc 2
đúng), rồi hỏi xác suất thấy chênh lệch lệch đến mức này nếu hai nấc thực ra tương đương.

In [ ]:
_prev_rows = load_stage("01_plain")
if _prev_rows is None:
    print("[SO SÁNH] ⚠ chưa có kết quả nấc '01_plain' → bỏ qua phần kiểm định.")
    print("          Chạy notebook nấc trước rồi quay lại cell này.")
else:
    _m_prev = pipeline.summarize(_prev_rows, "01_plain")
    print(f"\n{'═'*84}\n  NẤC 2 vs NẤC 1 — giá trị của prompt engineering\n{'═'*84}")
    print(f"{'nấc':<28}{'EA':>10}{'PA_strict':>12}{'PA_loose':>11}{'no_prog':>10}")
    for _n, _m in [("01_plain", _m_prev), ("02_prompt_eng", metrics)]:
        print(f"{_n:<28}{_m['EA']:>10.4f}{_m['PA_strict']:>12.4f}"
              f"{_m['PA_loose']:>11.4f}{_m['no_program']:>10.4f}")
    print(f"\n  Δ EA        = {metrics['EA'] - _m_prev['EA']:+.4f}")
    print(f"  Δ PA_strict = {metrics['PA_strict'] - _m_prev['PA_strict']:+.4f}")

    for _k in ("ea", "pa_strict"):
        stats.compare_pair(_prev_rows, rows, key=_k,
                           label="NẤC 2 vs NẤC 1 — giá trị của prompt engineering",
                           name_base="01_plain", name_variant="02_prompt_eng")

In [ ]:
_prev = load_stage("01_plain")
if _prev is not None:
    print(f"\n{'═'*80}\n  PROMPT ENGINEERING SỬA ĐƯỢC GÌ\n{'═'*80}")
    _fix = [(a, b) for a, b in zip(_prev, rows) if b["ea"] and not a["ea"]]
    _brk = [(a, b) for a, b in zip(_prev, rows) if a["ea"] and not b["ea"]]
    print(f"  Sửa đúng thêm : {len(_fix)}")
    print(f"  Làm hỏng      : {len(_brk)}")
    print(f"  Thực thu      : {len(_fix) - len(_brk):+d}")

    print(f"\n  ── Chuyển dịch kiểu lỗi ──")
    _a = Counter(r["outcome"] for r in _prev)
    _b = Counter(r["outcome"] for r in rows)
    print(f"  {'kết cục':<30}{'nấc 1':>8}{'nấc 2':>8}{'Δ':>8}")
    for _k in sorted(set(_a) | set(_b)):
        print(f"  {_k:<30}{_a.get(_k,0):>8}{_b.get(_k,0):>8}{_b.get(_k,0)-_a.get(_k,0):>+8}")
    print(f"\n  Chú ý hai dòng 'khong_co_program' và 'program_khong_chay_duoc':")
    print(f"  giảm mạnh nghĩa là phần quy tắc định dạng trong prompt đang có tác dụng.")

    print(f"\n  ── 5 ca prompt engineering sửa được ──")
    for _a_r, _b_r in _fix[:5]:
        print(f"\n  [{_b_r['id']}] {_b_r['question'][:84]}")
        print(f"    gold   : {_b_r['gold_program']}  = {_b_r['gold_answer']}")
        print(f"    nấc 1  : {_a_r['final_program'] or '(không có)'}  = {_a_r['pred_value']}")
        print(f"    nấc 2  : {_b_r['final_program']}  = {_b_r['pred_value']}")

## §6. Lưu kết quả

In [ ]:
save_stage("02_prompt_eng", rows, metrics,
           extra={"prompt_level": "engineered", "self_eval": False})
print(f"\n  EA = {metrics['EA']:.4f} | PA_strict = {metrics['PA_strict']:.4f} "
      f"| PA_loose = {metrics['PA_loose']:.4f}")
print("  → Nấc 3 (SFT) sẽ dùng ĐÚNG prompt này để dựng dữ liệu huấn luyện.")

## Kết luận nấc 2

Prompt ở nấc này trở thành **định dạng chuẩn** cho mọi nấc sau: nấc 3 fine-tune trên đúng
định dạng này, nấc 4 thêm bước tự soát lên trên nó, nấc 5 chèn bullet vào chính nó.

Nếu `PA_loose` ở đây thấp hơn nhiều so với mức ~51 % của mốc tham chiếu cho Qwen3-8B ở bước
sinh đầu tiên, hãy kiểm tra: phiên bản `transformers`/`vllm`, `temperature`, và dòng
`[PROMPT]` xem có bị cắt ngữ cảnh không.

**Tiếp theo:** `03_sft_qwen3.ipynb`.